# Earlier findings checked on dandiset 001538 version 0.261124.1858

This notebook belongs to the reconciliation of attempt zhai-2025-03 with attempt zhai-2025-02
(docs/protocol.md, "Repeat attempts"). zhai-2025-02 was made on the draft (modified 2026-11-09)
and zhai-2025-03 on version 0.261124.1858. Of the fifteen findings of zhai-2025-02, findings 1, 5,
6, 9, 13 and 14 are not reported by zhai-2025-03, and each is checked here directly on
[DANDI:001538](https://dandiarchive.org/dandiset/001538/0.261124.1858), version 0.261124.1858.
Each section opens with the earlier statement. The files are streamed with the entry's helper
module, `zhai.py`, and only the files each check needs are read. Findings 9 and 14 are about the
paper and are checked on the paper's per-cell tables (doi:10.5281/zenodo.17307059). The
conclusion for each finding is in the reconciliation block of `attempts/zhai-2025-03.yaml`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import zhai

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)
assets = zhai.list_assets()
T = zhai.paper_tables()
print("dandiset", zhai.DANDISET, "version", zhai.VERSION, "| assets", len(assets), "| NWB files", int(assets.nwb.sum()),
      "| subjects", assets.subject.nunique())
print("paper tables read:", len(T), "sheets")

## Earlier finding 1 (data_missing)

> Two cells with dendritic line scans were imaged at one dendritic site only, at two spots:
> 2016-05-26 cell 3 (CDGI knockout, off-state, Distal1 Spot1 and Spot2) and 2016-06-08 cell 1
> (on-state, Proximal1 Spot1 and Spot2). They have no proximal-distal pair and no index, while the
> paper counts them in Figures 3F and 7F (10 and 11 cells).

The line-scan sites of every cell in the two files, and the proximal-distal pairs with their
index, computed as in the main notebook.

In [ ]:
f1 = assets[assets.date.isin(["20160526", "20160608"]) & (assets.kind == "icephys+ophys")]
for _, a in f1.iterrows():
    h = zhai.open_h5(a.asset_id)
    t = zhai.recordings_table(h)
    t = t[t.site.str.contains("Prox|Dist")]
    print(a.path)
    print(t.groupby(["cell_id", "condition"]).site.apply(lambda s: sorted(set(s))).to_string())
    d = zhai.dendritic_file(a, h)
    print(d[["cell", "proximal", "distal", "index", "basal_ratio"]].round(3).to_string(), "\n")

## Earlier finding 5 (other, data)

> Mouse 7830 (Figure 10, control) has identical AIM sheets for sessions 4 and 5; the paper's table
> has the same values for both sessions.

The AIM scores of mouse 7830 per session, and the paper's Figure 10B totals for the mouse.

In [ ]:
f5 = assets[assets.nwb & assets.path.str.contains("M7830")]
aim = pd.concat([zhai.aim_file(a) for _, a in f5.iterrows()], ignore_index=True)
print("sessions:", aim.groupby("session").path.first().to_dict())
w = aim.pivot_table(index=["behavior", "timestamp"], columns="session", values="score")
print(w.to_string())
print("\nsessions 4 and 5 identical at every time point and behaviour:", bool(w[4].equals(w[5])))
tot = aim[aim.behavior.isin(["axial", "limb", "orolingual"])].groupby("session").score.sum()
p10 = T["Fig 10B_total AIMs"].set_axis(["mouse", "group", "session", "total"], axis=1)
p10 = p10[p10.mouse.astype(str).str.contains("7830")]
print("session totals, dandiset:", tot.to_dict())
print("session totals, paper:", dict(zip(p10.session, p10.total)))

## Earlier finding 6 (metadata_insufficient)

> Subjects are defined per recording day. Mouse counts differ from the paper's legends in four
> groups (Sr2+-oEPSC dSPN and iSPN off-state, 3 subjects against 4 mice; M1R CRISPR control and
> deletion somatic cells, 2 and 3 against 3 and 4), which mice sharing a recording day would
> explain; the files do not say which mouse each cell came from.

Cells and subjects of these four groups, read from the files' recordings tables, and the subject
description of one file. The paper's legends give 10 cells from four mice (Figure 2I, off-state),
7 cells from four mice (Figure 4H, off-state), and 6 cells from three mice and 8 cells from four
mice (Figure 9D, control and M1R deletion).

In [ ]:
f6 = assets[(assets.kind == "icephys") & (((assets.group == "OffState") & assets.date.str.startswith("2023"))
                                          | assets.date.isin(["20221004", "20221005", "20221012", "20221129", "20221130"]))]
rows = []
for _, a in f6.iterrows():
    h = zhai.open_h5(a.asset_id)
    t = zhai.recordings_table(h)
    kind = np.where(t.r_series.str.contains("VoltageClamp"), "Sr2+-oEPSC", "somatic")
    for (k, cond), g in t.assign(kind=kind).groupby(["kind", "condition"]):
        rows.append(dict(cell_type=a.cell_type, kind=k, condition=cond, subject=a.subject, cells=g.cell_id.nunique()))
    if a.date == "20221004":
        s = h["general/subject"]
        print("subject_id:", zhai._str(s["subject_id"][()]), "| description:", zhai._str(s["description"][()]), "\n")
g6 = pd.DataFrame(rows)
print(g6.groupby(["cell_type", "kind", "condition"]).agg(cells=("cells", "sum"), subjects=("subject", "nunique")).to_string())

## Earlier finding 9 (other, paper)

> The per-cell Sr2+-oEPSC amplitudes of Figures 2H and 4G, labelled average amplitude in the
> paper's tables, equal the median of each cell's event list in the Figure 2G and 4F sheets for all
> 33 cells and the mean for none.

The paper's per-cell amplitudes against the mean and the median of each cell's events in its own
event lists.

In [ ]:
ev = pd.concat([T["Fig 2G_dSPN Sr-oEPSC histogram"].set_axis(["id", "group", "ev", "amp"], axis=1),
                T["Fig 4F_iSPN Sr-oEPSC histogram"].set_axis(["id", "group", "ev", "amp"], axis=1)])
pc = pd.concat([T["Fig 2H_dSPN Sr-oEPSC amp"].set_axis(["id", "group", "paper"], axis=1),
                T["Fig 4G_iSPN Sr-oEPSC amp"].set_axis(["id", "group", "paper"], axis=1)])
s = ev.groupby("id").amp.agg(["mean", "median", "size"]).reset_index().merge(pc, on="id")
s["eq_median"] = np.isclose(s["median"], s.paper, atol=0.006)
s["eq_mean"] = np.isclose(s["mean"], s.paper, atol=0.006)
print("cells:", len(s), "| equal to the median (within 0.006 pA):", int(s.eq_median.sum()),
      "| equal to the mean:", int(s.eq_mean.sum()))
print(s.round(3).to_string())

## Earlier finding 13 (metadata_error)

> The session names of the SCH 23390 and sulpiride files (D1RaSch, D2RaSul) state a drug, while 11
> somatic cells in these files have baseline on-state sweeps and 4 of them are baseline only (dSPN
> 2019-05-07 cell 1, iSPN 2019-04-08 cell 1, 2019-04-09 cell 1 and 2019-05-08 cell 2). Each sweep's
> group is in the condition and drug columns of the intracellular recordings table; grouping by
> session name puts these cells in the drug groups.

The somatic cells of the files whose session name states a drug, with the conditions their
sweeps carry.

In [ ]:
rows = []
for _, a in assets[assets.drug.isin(["D1RaSch", "D2RaSul"]) & assets.nwb].iterrows():
    h = zhai.open_h5(a.asset_id)
    t = zhai.recordings_table(h)
    t = t[(t.site == "Soma") & t.r_series.str.contains("CurrentClamp")]
    for c, g in t.groupby("cell_id"):
        rows.append(dict(path=a.path.split("_ses-")[1], date=a.date, cell=c, name_drug=a.drug,
                         conditions=sorted(set(g.condition)), drugs=sorted(set(g.drug.astype(str)))))
g13 = pd.DataFrame(rows)
g13["baseline"] = g13.conditions.apply(lambda c: "LID on-state" in c)
g13["baseline_only"] = g13.conditions.apply(lambda c: c == ["LID on-state"])
print(g13.to_string())
print("\nsomatic cells in drug-named files:", len(g13), "| with baseline on-state sweeps:", int(g13.baseline.sum()),
      "| baseline only:", int(g13.baseline_only.sum()))

## Earlier finding 14 (paper_inconsistency)

> Recomputed on the paper's per-cell tables (doi:10.5281/zenodo.17307059), two reported P values
> differ. The Wilcoxon test of oxo-M in CDGI KO iSPNs (Figure 7E, 12 cells) gives P = 0.4697 where
> the paper reports 0.910, and the Mann-Whitney test of rotations in Figure 8A (10 KO and 12
> wild-type mice) gives P = 0.01198 where the paper reports 0.0092. The wild-type oxo-M test gives
> P = 0.009766, as the paper's 0.0098.

The same tests on the same tables, which do not depend on the dandiset's version.

In [ ]:
e7 = T["Fig 7E_oxoM effect on DE"].set_axis(["id", "genotype", "treatment", "v"], axis=1)
for gt in ["WT", "KO"]:
    w = e7[e7.genotype == gt].pivot_table(index="id", columns="treatment", values="v")
    print(f"Figure 7E {gt}: n = {len(w)}, Wilcoxon P = {stats.wilcoxon(w['before (control)'], w['oxoM']).pvalue:.4g}")
r8 = T["Fig 8A_rotations 5th"].set_axis(["mouse", "genotype", "rot"], axis=1)
ko, wt = r8[r8.genotype != "WT"].rot, r8[r8.genotype == "WT"].rot
print("Figure 8A groups:", r8.genotype.value_counts().to_dict())
print(f"Figure 8A: Mann-Whitney P = {stats.mannwhitneyu(ko, wt, alternative='two-sided').pvalue:.4g}")